# Human Pose Estimation — OpenPose BODY_25 via `cv2.dnn`

Estimating human body pose with the BODY_25 model: heatmaps, Part Affinity
Fields, and both the bottom-up and the top-down pipeline.

All paths are relative to the project root, so Jupyter has to be started from
there. Kernel: **Python 3.11 (Computer Vision venv)**.

## What the network returns

BODY_25 is a fully convolutional Caffe graph. It takes an image and returns a
tensor of shape `(1, 78, H/8, W/8)`. The stride of 8 comes from four poolings in
the VGG-like backbone: every output cell covers an 8x8 square of the input.

Those 78 channels are three different things packed into one tensor:

| channels | what it is | how to read it |
|---|---|---|
| 0–24 | heatmaps of the 25 joints | value = probability that the joint is here |
| 25 | background | not needed, but it occupies a channel |
| 26–77 | PAF (Part Affinity Fields) | 26 limbs x 2 channels: the x and y component of a unit vector along the limb |

Heatmaps answer *where the joints are*, PAFs answer *which joints belong to the
same person*. The second question is why PAFs exist at all: with three people in
frame, the left-wrist heatmap has three peaks, and nothing else tells you whose
is whose.

Two approaches to keep apart:

- **bottom-up** — run the whole frame once, find every joint, group them with
  PAFs. Runtime does not depend on the number of people.
- **top-down** — run a person detector first, then pose on each crop. The crop is
  stretched to the network input, so small people "grow"; you pay per person.

---

## Block 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import urllib.request          # a bare `import urllib` does NOT give you this submodule
import matplotlib.pyplot as plt
import time

%matplotlib inline

---

## Block 2. Downloading the model, with a size check

Pulls the architecture (`pose_deploy.prototxt`, ~42 KB) and the weights
(`pose_iter_584000.caffemodel`, exactly 104,715,850 bytes) into
`models/pose_body25/`.

Why not a one-line `urlretrieve`:

1. The official CMU host (`posefs1.perception.cs.cmu.edu`) stopped resolving
   long ago, which kills every tutorial that uses it. The architecture comes
   from CMU's GitHub, the weights from a Hugging Face mirror.
2. 100 MB over a flaky link gets cut halfway. On a broken connection
   `urlretrieve` leaves a **truncated file that looks perfectly normal**:
   `os.path.isfile()` says `True`, the next run skips the download, and
   `readNetFromCaffe` then dies with an unreadable protobuf error.

So the function below resumes with a `Range` header and verifies the final size
against a known number instead of trusting that the file exists.

In [ ]:
MODEL_DIR = "models/pose_body25"
protoFile = os.path.join(MODEL_DIR, "pose_deploy.prototxt")
weightsFile = os.path.join(MODEL_DIR, "pose_iter_584000.caffemodel")

PROTO_URL = ("https://raw.githubusercontent.com/CMU-Perceptual-Computing-Lab/"
             "openpose/master/models/pose/body_25/pose_deploy.prototxt")
WEIGHTS_URL = "https://huggingface.co/dylanholmes/openpose-caffemodels/resolve/main/body25.caffemodel"
WEIGHTS_SIZE = 104_715_850     # anything smaller means the download was cut short


def download(url, dst, expected_size=None, attempts=10, timeout=60):
    """Download `url` into `dst`, resuming after a dropped connection."""
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)

    for attempt in range(1, attempts + 1):
        have = os.path.getsize(dst) if os.path.isfile(dst) else 0
        if expected_size and have >= expected_size:
            return True
        if expected_size is None and have:
            return True

        # Ask the server to continue from the byte the last attempt stopped at.
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as resp:
                # 206 = the server honoured Range, so appending is safe.
                # 200 = it ignored Range and resends from byte 0, so appending
                #       would corrupt the file - reset and overwrite instead.
                mode = "ab" if (have and resp.status == 206) else "wb"
                if mode == "wb":
                    have = 0
                total = have + int(resp.headers.get("Content-Length", 0))
                reported = have
                with open(dst, mode) as f:
                    while True:
                        chunk = resp.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
                        have += len(chunk)
                        # Report every 10 MB, not every chunk: "\r" does not
                        # erase the line in Jupyter the way it does in a
                        # terminal, so each print would become its own line.
                        if have - reported >= 10 << 20:
                            reported = have
                            print(f"  {os.path.basename(dst)}: {have/1e6:6.1f} / {total/1e6:.1f} MB")
        except Exception as exc:
            # A dropped connection is the normal case here, not a failure.
            print(f"  attempt {attempt} interrupted: {exc}")
            time.sleep(2)

    # Trust the size, not the mere existence of the file.
    return os.path.isfile(dst) and (not expected_size or os.path.getsize(dst) >= expected_size)


download(PROTO_URL, protoFile)
download(WEIGHTS_URL, weightsFile, WEIGHTS_SIZE)
print("prototxt:", os.path.getsize(protoFile), "bytes")
print("weights :", os.path.getsize(weightsFile), "bytes  (expected", WEIGHTS_SIZE, ")")

---

## Block 3. Loading the network

Two things worth knowing here. The argument order of `readNetFromCaffe` is
**prototxt first, weights second** (`readNetFromTensorflow` is the other way
round) — swap them and you get a protobuf error that looks exactly like the one
a truncated file gives.

And CPU is not a compromise here: there is no CUDA on a Mac, `cv2.dnn` does not
use Metal, and for this network the OpenCL target is usually slower than CPU.

In [ ]:
net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)     # prototxt first, then weights
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

print("layers:", len(net.getLayerNames()))                 # 261

---

## Block 4. BODY_25 constants

Three lists, and all three have to agree with each other.

**Joints** — the index is the heatmap channel:

```
0 Nose      5 LShoulder  10 RKnee    15 REye   20 LSmallToe
1 Neck      6 LElbow     11 RAnkle   16 LEye   21 LHeel
2 RShoulder 7 LWrist     12 LHip     17 REar   22 RBigToe
3 RElbow    8 MidHip     13 LKnee    18 LEar   23 RSmallToe
4 RWrist    9 RHip       14 LAnkle   19 LBigToe 24 RHeel
```

R/L are from the point of view of **the person in the frame**, not the viewer.

`POSE_PAIRS` holds the 26 limbs the network computes PAFs for, and `MAP_IDX`
holds the two PAF channels of each limb relative to `PAF_OFFSET`. Both come from
`poseParameters.cpp` in OpenPose and their order is locked together:
`MAP_IDX[k]` are the channels for `POSE_PAIRS[k]`. Reorder either one and the
skeleton will connect a knee to an ear — a silent failure, with no exception.

In [ ]:
# Index = heatmap channel. R/L are from the point of view of the person in frame.
BODY_PARTS = ["Nose", "Neck", "RShoulder", "RElbow", "RWrist", "LShoulder",
              "LElbow", "LWrist", "MidHip", "RHip", "RKnee", "RAnkle", "LHip",
              "LKnee", "LAnkle", "REye", "LEye", "REar", "LEar", "LBigToe",
              "LSmallToe", "LHeel", "RBigToe", "RSmallToe", "RHeel"]

N_POINTS = len(BODY_PARTS)     # 25
PAF_OFFSET = N_POINTS + 1      # 0-24 are joints, 25 is background, 26+ are PAFs

In [ ]:
# The order of these two lists is locked together: MAP_IDX[k] are the PAF
# channels of POSE_PAIRS[k]. Reorder one of them and the skeleton connects a
# knee to an ear - silently, without raising anything.
POSE_PAIRS = [[1, 8], [1, 2], [1, 5], [2, 3], [3, 4], [5, 6], [6, 7], [8, 9],
              [9, 10], [10, 11], [8, 12], [12, 13], [13, 14], [1, 0], [0, 15],
              [15, 17], [0, 16], [16, 18], [2, 17], [5, 18], [14, 19], [19, 20],
              [14, 21], [11, 22], [22, 23], [11, 24]]

MAP_IDX = [[0, 1], [14, 15], [22, 23], [16, 17], [18, 19], [24, 25], [26, 27],
           [6, 7], [2, 3], [4, 5], [8, 9], [10, 11], [12, 13], [30, 31],
           [32, 33], [36, 37], [34, 35], [38, 39], [20, 21], [28, 29], [40, 41],
           [42, 43], [44, 45], [46, 47], [48, 49], [50, 51]]

# Pairs [2,17] and [5,18] (shoulder-to-ear): useful for grouping people, but on
# the drawing they strike a line straight across the face.
SKIP_RENDER = {18, 19}

# One BGR colour per limb, spread evenly around the hue circle.
COLORS = [tuple(int(v) for v in cv2.cvtColor(
              np.uint8([[[i * 180 // len(POSE_PAIRS), 255, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])
          for i in range(len(POSE_PAIRS))]

**Structural check.** The anatomy check comes in block 7, once `draw_pose`
exists — but this one is free and catches half the mistakes right here. The
second assert is the useful one: a duplicated or missing PAF channel fails now
instead of showing up as a crooked skeleton three blocks later.

In [ ]:
assert len(POSE_PAIRS) == len(MAP_IDX) == 26
assert sorted(c for pair in MAP_IDX for c in pair) == list(range(52))   # every PAF channel exactly once
assert max(j for pair in POSE_PAIRS for j in pair) == N_POINTS - 1      # no joint index above 24
assert len(COLORS) == len(POSE_PAIRS)
print("constants OK")

A small helper used everywhere below — matplotlib expects RGB, OpenCV gives BGR.

In [ ]:
def show(im, title="", figsize=(11, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(im[:, :, ::-1])      # BGR -> RGB for matplotlib
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

---

## Block 5. One forward pass

Every argument of `blobFromImage` has a reason:

- `1/255.0` — the model was trained on inputs in the range [0, 1];
- `(in_width, in_height)` — the width follows the frame's aspect ratio and is
  rounded to a multiple of 8. Break the ratio and the person is squeezed
  horizontally, which costs accuracy; skip the rounding and OpenCV pads the size
  itself, shifting the maps by half a cell;
- `(0, 0, 0)` — no mean subtraction, the model does not expect it;
- **`swapRB=False`** — the Caffe model was trained on BGR and `cv2.imread`
  already returns BGR. Set it to `True` (as TensorFlow models want) and the
  network still finds *something*, just worse and less reliably. A classic
  silent bug;
- `crop=False` — otherwise OpenCV centre-crops the frame instead of resizing it.

The last line stretches all 78 maps back to frame size, so peak coordinates are
already in original pixels and nothing below has to multiply by `w/out_w`. The
price is memory: 78 float maps at full frame size.

In [ ]:
def run_pose(net, im, in_height=368):
    h, w = im.shape[:2]
    # Width follows the frame aspect ratio, rounded to a multiple of the
    # network stride (8).
    in_width = max(8, int(round(in_height * w / h / 8)) * 8)

    blob = cv2.dnn.blobFromImage(im, 1 / 255.0, (in_width, in_height),
                                 (0, 0, 0), swapRB=False, crop=False)

    net.setInput(blob)
    out = net.forward()             # (1, 78, in_height/8, in_width/8)

    # Stretch every map back to frame size, so peaks are already in the pixel
    # coordinates of the original image.
    return np.stack([cv2.resize(out[0, i], (w, h)) for i in range(out.shape[1])])

**Visual check.** Expect `~0.7–1.2 s` and `maps.shape == (78, 342, 548)`.

Nothing is drawn on the frame yet — `run_pose` does not modify `im`, and
`draw_pose` only appears in block 7. What this cell proves is that the maps come
back in frame pixels and that the image itself loaded correctly.

In [ ]:
im = cv2.imread("data/images/messi5.jpg")

t = time.time()
maps = run_pose(net, im)
print(f"{time.time() - t:.2f} s   im.shape={im.shape}   maps.shape={maps.shape}")

# The real check here. Without the cv2.resize inside run_pose the shape would be
# (78, 46, 69), and every coordinate computed below would be off by a factor of 8.
assert maps.shape == (78,) + im.shape[:2]

# Plain input frame - no lines yet. This only confirms the image decoded and the
# BGR->RGB swap in show() is right (the grass has to look green, not red).
show(im, "input frame")

In [44]:
def keypoints_single(maps, threshold=0.1):
    points = []
    for i in range(N_POINTS):
        _, conf, _, loc = cv2.minMaxLoc(maps[i])
        points.append((loc[0], loc[1], conf) if conf > threshold else None)
    return points